# Lesson 3A: DBSCAN - Density-Based Spatial Clustering

## The Geographical Clustering Challenge 🗺️

You're a data scientist at Uber working on **driver positioning strategy**. You have GPS coordinates of thousands of ride requests across San Francisco. Your goal: identify **high-demand zones** where drivers should wait for pickups.

But traditional clustering (K-Means, Hierarchical) **fails**:
- **K-Means problem**: Requires specifying k upfront - you don't know how many zones exist!
- **Spherical assumption**: K-Means creates circular zones, but demand follows streets, neighborhoods, and irregular geographic patterns
- **Noise sensitivity**: Some requests are random outliers (middle of nowhere, parks, water) that shouldn't define a zone

**You need a clustering method that:**
1. ✅ **Discovers clusters automatically** (no need to specify k)
2. ✅ **Handles arbitrary shapes** (follows streets, coastlines)
3. ✅ **Identifies outliers** (noise points that don't belong to any cluster)
4. ✅ **Based on density** (high-demand zones = dense regions)

**Enter DBSCAN**: Density-Based Spatial Clustering of Applications with Noise

## Real-World Applications

🗺️ **Geographic Clustering**
- Uber/Lyft: High-demand pickup zones
- Crime analysis: Hot spots for police deployment
- Earthquake epicenters: Fault line detection

🏙️ **Urban Planning**
- Identify neighborhoods from building density
- Public transit station placement
- Traffic congestion zones

🔬 **Scientific Research**
- Astronomy: Galaxy cluster discovery
- Biology: Cell colony detection in microscopy
- Climate: Storm pattern identification

💻 **Anomaly Detection**
- Network intrusion: Unusual traffic patterns (outliers = attacks)
- Manufacturing: Defect detection
- Fraud: Unusual transaction clusters

## What You'll Learn

1. ✅ **Core concepts**: Dense regions, ε-neighborhoods, core/border/noise points
2. ✅ **DBSCAN algorithm**: From-scratch implementation
3. ✅ **Parameter tuning**: How to choose ε (epsilon) and MinPts
4. ✅ **Comparison with K-Means**: Strengths and weaknesses
5. ✅ **Visualization**: See how DBSCAN discovers arbitrary shapes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.preprocessing import StandardScaler
from collections import deque

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries loaded successfully!")

## Key Concepts

### 1. Density-Based Clustering

**Core idea**: Clusters are **dense regions** separated by **low-density** regions.

**Intuition**: 
- In a high-demand Uber zone, ride requests are **close together** (dense)
- Between zones, requests are **sparse** (low density)
- Random requests in parks/water are **isolated** (noise)

###  2. ε-Neighborhood (Epsilon-Neighborhood)

**Definition**: For a point p, the **ε-neighborhood** is the set of all points within distance ε from p.

$$N_\varepsilon(p) = \{q \in D \ | \ \text{dist}(p, q) \leq \varepsilon\}$$

**Intuition**: Draw a circle of radius ε around p - all points inside are neighbors.

### 3. Three Types of Points

Given ε (radius) and MinPts (minimum points):

#### **Core Point** 🔴
- Has **≥ MinPts** points in its ε-neighborhood (including itself)
- Forms the "backbone" of clusters
- Example: Dense pickup zone center

$$|N_\varepsilon(p)| \geq \text{MinPts}$$

#### **Border Point** 🟡
- Has **< MinPts** points in its ε-neighborhood
- BUT is within ε of a core point
- On the "edge" of clusters
- Example: Pickup at edge of high-demand zone

#### **Noise Point** ⚫
- Has **< MinPts** neighbors
- NOT within ε of any core point
- Outlier, doesn't belong to any cluster
- Example: Random pickup request in the ocean

### 4. Density-Reachability

**Directly density-reachable**:
- Point q is directly density-reachable from p if:
  1. p is a core point
  2. q is in N_ε(p)

**Density-reachable** (transitive):
- q is density-reachable from p if there's a chain:
  p → p₁ → p₂ → ... → q
  where each step is directly density-reachable

**Density-connected**:
- p and q are density-connected if there exists a point o such that both p and q are density-reachable from o

**Cluster**: Maximal set of density-connected points

In [ ]:
# Visualize the concepts with a simple example
from matplotlib.patches import Circle

# Create simple dataset
np.random.seed(42)
points = np.array([
    [1, 1], [1.2, 1.1], [1.1, 0.9], [0.9, 1.2],  # Dense cluster
    [5, 5], [5.1, 5.2], [4.9, 4.8],               # Another dense cluster  
    [3, 3],                                        # Border point
    [7, 1], [1, 7]                                 # Noise points
])

# Parameters
eps = 0.5
min_pts = 3

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Plot points
for i, (x, y) in enumerate(points):
    n_neighbors = np.sum(np.linalg.norm(points - points[i], axis=1) <= eps)
    
    if n_neighbors >= min_pts:
        # Core point
        color, marker, label = 'red', 'o', 'Core'
        size = 200
    elif any(np.linalg.norm(points - points[i], axis=1) <= eps) > 0:
        # Check if near a core point
        is_border = False
        for j, (px, py) in enumerate(points):
            if i != j:
                n_j = np.sum(np.linalg.norm(points - points[j], axis=1) <= eps)
                if n_j >= min_pts and np.linalg.norm(points[i] - points[j]) <= eps:
                    is_border = True
                    break
        if is_border:
            color, marker, label = 'yellow', 's', 'Border'
            size = 150
        else:
            color, marker, label = 'gray', 'x', 'Noise'
            size = 150
    else:
        color, marker, label = 'gray', 'x', 'Noise'
        size = 150
    
    ax.scatter(x, y, c=color, marker=marker, s=size, edgecolors='black', linewidths=2,
              label=label if i == 0 or (label == 'Border' and i == 7) or (label == 'Noise' and i == 8) else '')

# Draw epsilon circles around core points
for i, (x, y) in enumerate(points):
    n_neighbors = np.sum(np.linalg.norm(points - points[i], axis=1) <= eps)
    if n_neighbors >= min_pts:
        circle = Circle((x, y), eps, fill=False, edgecolor='red', linestyle='--', linewidth=2, alpha=0.5)
        ax.add_patch(circle)

ax.set_xlabel('X', fontweight='bold', fontsize=12)
ax.set_ylabel('Y', fontweight='bold', fontsize=12)
ax.set_title(f'DBSCAN Concepts: Core, Border, and Noise Points\n(ε={eps}, MinPts={min_pts})',
            fontweight='bold', fontsize=14)
ax.legend(fontsize=12, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print("✅ Visualization complete!")
print(f"\nParameters: ε={eps}, MinPts={min_pts}")
print("• Red circles (○): Core points - dense regions")
print("• Yellow squares (□): Border points - cluster edges")
print("• Gray X's (×): Noise points - outliers")
print("• Dashed circles: ε-neighborhoods of core points")

## The DBSCAN Algorithm

### Pseudocode

```python
DBSCAN(D, eps, MinPts):
    C = 0  # Cluster counter
    labels = [UNVISITED for all points]
    
    for each point P in D:
        if P is labeled:
            continue
            
        # Find neighbors
        neighbors = regionQuery(P, eps)
        
        if |neighbors| < MinPts:
            labels[P] = NOISE
        else:
            C = C + 1  # New cluster
            expandCluster(P, neighbors, C, eps, MinPts)
    
    return labels

expandCluster(P, neighbors, C, eps, MinPts):
    labels[P] = C  # Add P to cluster C
    queue = neighbors
    
    while queue not empty:
        Q = queue.pop()
        
        if labels[Q] == NOISE:
            labels[Q] = C  # Border point
        
        if labels[Q] is not UNVISITED:
            continue
            
        labels[Q] = C
        Q_neighbors = regionQuery(Q, eps)
        
        if |Q_neighbors| >= MinPts:  # Q is core point
            queue.extend(Q_neighbors)
```

### Key Steps

1. **Mark all points as UNVISITED**
2. **For each UNVISITED point P**:
   - Find its ε-neighborhood
   - If |neighbors| < MinPts → mark as NOISE
   - Else → start new cluster, expand it
3. **Expand cluster**:
   - Add P to cluster
   - For each neighbor Q:
     - If NOISE, change to border point
     - If UNVISITED, add to cluster
     - If Q is core point, add its neighbors to queue

### Time Complexity

- **With spatial index** (KD-tree, Ball tree): **O(n log n)**
- **Without index** (naive): **O(n²)**

Where n = number of points

In [ ]:
# DBSCAN from scratch!
class DBSCAN:
    """
    DBSCAN: Density-Based Spatial Clustering of Applications with Noise
    
    From-scratch implementation for educational purposes.
    """
    
    def __init__(self, eps=0.5, min_pts=5):
        """
        Parameters:
            eps: Maximum distance between two samples for one to be considered
                 as in the neighborhood of the other (epsilon)
            min_pts: Minimum number of samples in a neighborhood for a point
                     to be considered as a core point
        """
        self.eps = eps
        self.min_pts = min_pts
        self.labels_ = None
        
    def fit(self, X):
        """
        Perform DBSCAN clustering.
        
        Args:
            X: Data of shape (n_samples, n_features)
        """
        n_samples = X.shape[0]
        
        # Initialize all points as unvisited (-2)
        labels = np.full(n_samples, -2, dtype=int)
        
        cluster_id = 0
        
        for point_id in range(n_samples):
            # Skip if already processed
            if labels[point_id] != -2:
                continue
            
            # Find epsilon-neighborhood
            neighbors = self._region_query(X, point_id)
            
            # If not enough neighbors, mark as noise (-1)
            if len(neighbors) < self.min_pts:
                labels[point_id] = -1
            else:
                # Start new cluster
                self._expand_cluster(X, labels, point_id, neighbors, cluster_id)
                cluster_id += 1
        
        self.labels_ = labels
        return self
    
    def _region_query(self, X, point_id):
        """
        Find all points in epsilon-neighborhood of point.
        
        Returns:
            List of indices of neighbor points
        """
        distances = np.linalg.norm(X - X[point_id], axis=1)
        return np.where(distances <= self.eps)[0].tolist()
    
    def _expand_cluster(self, X, labels, point_id, neighbors, cluster_id):
        """
        Expand cluster by adding all density-reachable points.
        """
        # Add seed point to cluster
        labels[point_id] = cluster_id
        
        # Use queue for breadth-first search
        queue = deque(neighbors)
        
        while queue:
            current_point = queue.popleft()
            
            # Change noise to border point
            if labels[current_point] == -1:
                labels[current_point] = cluster_id
            
            # Skip if already in a cluster
            if labels[current_point] != -2:
                continue
            
            # Add to cluster
            labels[current_point] = cluster_id
            
            # Find neighbors of current point
            current_neighbors = self._region_query(X, current_point)
            
            # If core point, add its neighbors to queue
            if len(current_neighbors) >= self.min_pts:
                queue.extend(current_neighbors)

print("✅ DBSCAN class implemented from scratch!")
print("   Methods: fit(), _region_query(), _expand_cluster()")
print("   Attributes: labels_ (cluster assignments)")

In [ ]:
# Test DBSCAN on different datasets

# Create three challenging datasets
datasets = {
    'Moons': make_moons(n_samples=300, noise=0.05, random_state=42)[0],
    'Circles': make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)[0],
    'Anisotropic Blobs': make_blobs(n_samples=300, centers=3, random_state=42)[0]
}

# Add noise to blobs and transform
X_blobs = datasets['Anisotropic Blobs']
transformation = [[0.6, -0.6], [-0.4, 0.8]]
X_aniso = np.dot(X_blobs, transformation)
datasets['Anisotropic Blobs'] = X_aniso

# Standardize
for name in datasets:
    scaler = StandardScaler()
    datasets[name] = scaler.fit_transform(datasets[name])

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Row 1: K-Means (for comparison)
from sklearn.cluster import KMeans

for idx, (name, X) in enumerate(datasets.items()):
    ax = axes[0, idx]
    kmeans = KMeans(n_clusters=2, random_state=42)
    y_kmeans = kmeans.fit_predict(X)
    
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y_kmeans, cmap='viridis', 
                        s=50, alpha=0.6, edgecolors='k')
    ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
              c='red', s=300, marker='*', edgecolors='black', linewidths=2)
    ax.set_title(f'K-Means on {name}', fontweight='bold', fontsize=12)
    ax.set_xlabel('Feature 1', fontweight='bold')
    ax.set_ylabel('Feature 2', fontweight='bold')
    ax.grid(True, alpha=0.3)

# Row 2: DBSCAN
for idx, (name, X) in enumerate(datasets.items()):
    ax = axes[1, idx]
    
    # Use our from-scratch DBSCAN
    dbscan = DBSCAN(eps=0.3, min_pts=5)
    dbscan.fit(X)
    y_dbscan = dbscan.labels_
    
    # Plot clusters
    unique_labels = set(y_dbscan)
    colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
    
    for label, color in zip(unique_labels, colors):
        if label == -1:
            # Noise points in black
            color = 'black'
            marker = 'x'
            size = 50
        else:
            marker = 'o'
            size = 50
        
        mask = y_dbscan == label
        ax.scatter(X[mask, 0], X[mask, 1], c=[color], marker=marker,
                  s=size, alpha=0.6, edgecolors='k',
                  label=f'Cluster {label}' if label != -1 else 'Noise')
    
    ax.set_title(f'DBSCAN on {name}', fontweight='bold', fontsize=12)
    ax.set_xlabel('Feature 1', fontweight='bold')
    ax.set_ylabel('Feature 2', fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
    
    # Count stats
    n_clusters = len(set(y_dbscan)) - (1 if -1 in y_dbscan else 0)
    n_noise = list(y_dbscan).count(-1)
    ax.text(0.02, 0.98, f'{n_clusters} clusters\n{n_noise} noise',
           transform=ax.transAxes, fontsize=10, verticalalignment='top',
           bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('K-Means vs DBSCAN: Handling Arbitrary Shapes',
            fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

print("✅ Comparison complete!")
print("\nKey Observations:")
print("• K-Means: Assumes spherical clusters, fails on moons/circles")
print("• DBSCAN: Successfully finds arbitrary shapes!")
print("• DBSCAN identifies noise points (shown as black X's)")
print("• No need to specify k - DBSCAN discovers the number automatically")

## Parameter Selection: ε and MinPts

### Challenge
DBSCAN requires two parameters:
- **ε (epsilon)**: Maximum distance between two samples to be neighbors
- **MinPts**: Minimum number of points to form a dense region

### Rules of Thumb

**MinPts:**
- Common choice: **MinPts = 2 × dimensions** (for 2D: MinPts = 4)
- Higher dimensions: **MinPts ≥ dimensions + 1**
- Larger MinPts → fewer, denser clusters
- Smaller MinPts → more, looser clusters

**ε (epsilon):**
- Use **k-distance graph** (elbow method for DBSCAN)
- For each point, compute distance to k-th nearest neighbor (k = MinPts)
- Sort these distances
- Plot: sorted point index vs k-distance
- **Elbow** in the plot → good ε value

### K-Distance Plot Method

1. For each point, find distance to MinPts-th nearest neighbor
2. Sort these distances ascending
3. Plot sorted distances
4. Look for "elbow" - sharp change in slope
5. ε = distance at elbow point

In [ ]:
# Demonstrate k-distance plot for epsilon selection
from sklearn.neighbors import NearestNeighbors

# Use moons dataset
X_moons = make_moons(n_samples=300, noise=0.05, random_state=42)[0]
X_moons = StandardScaler().fit_transform(X_moons)

# Compute k-distance for k=4 (MinPts=5, so 4 neighbors)
k = 4
nbrs = NearestNeighbors(n_neighbors=k+1).fit(X_moons)  # +1 because includes self
distances, indices = nbrs.kneighbors(X_moons)

# Get distance to k-th neighbor (index k because 0 is self)
k_distances = distances[:, k]
k_distances_sorted = np.sort(k_distances)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: K-distance graph
ax = axes[0]
ax.plot(range(len(k_distances_sorted)), k_distances_sorted, 'b-', linewidth=2)
ax.set_xlabel('Points (sorted by distance)', fontweight='bold', fontsize=12)
ax.set_ylabel(f'{k}-Distance', fontweight='bold', fontsize=12)
ax.set_title('K-Distance Graph for ε Selection', fontweight='bold', fontsize=14)
ax.grid(True, alpha=0.3)

# Mark potential epsilon value (at elbow)
elbow_idx = np.where(k_distances_sorted > 0.25)[0][0]  # First point > threshold
epsilon_suggestion = k_distances_sorted[elbow_idx]
ax.axhline(y=epsilon_suggestion, color='red', linestyle='--', linewidth=2,
          label=f'Suggested ε ≈ {epsilon_suggestion:.3f}')
ax.legend(fontsize=11)

# Plot 2: DBSCAN result with suggested epsilon
ax = axes[1]
dbscan_tuned = DBSCAN(eps=epsilon_suggestion, min_pts=5)
dbscan_tuned.fit(X_moons)
y_pred = dbscan_tuned.labels_

unique_labels = set(y_pred)
colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))

for label, color in zip(unique_labels, colors):
    if label == -1:
        color = 'black'
        marker = 'x'
    else:
        marker = 'o'
    
    mask = y_pred == label
    ax.scatter(X_moons[mask, 0], X_moons[mask, 1], c=[color], marker=marker,
              s=50, alpha=0.7, edgecolors='k',
              label=f'Cluster {label}' if label != -1 else 'Noise')

ax.set_xlabel('Feature 1', fontweight='bold', fontsize=12)
ax.set_ylabel('Feature 2', fontweight='bold', fontsize=12)
ax.set_title(f'DBSCAN Result (ε={epsilon_suggestion:.3f}, MinPts=5)',
            fontweight='bold', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

n_clusters = len(set(y_pred)) - (1 if -1 in y_pred else 0)
n_noise = list(y_pred).count(-1)

print("✅ K-distance plot method complete!")
print(f"\nSuggested ε: {epsilon_suggestion:.3f}")
print(f"Result: {n_clusters} clusters found, {n_noise} noise points")
print("\nHow to read k-distance plot:")
print("• X-axis: Points sorted by distance to k-th neighbor")
print("• Y-axis: Distance to k-th neighbor")
print("• Elbow point: Sharp increase indicates good ε value")
print("• Points after elbow: Become noise with this ε")

## DBSCAN vs K-Means: When to Use Each

### Comparison Table

| Feature | DBSCAN | K-Means |
|---------|--------|---------|
| **Number of clusters** | Discovers automatically | Must specify k |
| **Cluster shape** | Arbitrary (non-convex) | Spherical only |
| **Noise handling** | Identifies outliers | Every point assigned |
| **Density variation** | Struggles with varying densities | Handles well |
| **Scalability** | O(n log n) with index | O(nki) |
| **Parameter sensitivity** | Sensitive to ε and MinPts | Sensitive to k, initialization |
| **Deterministic** | Yes | No (random initialization) |

### ✅ Use DBSCAN When:

1. **Unknown number of clusters**
   - Don't know k in advance
   - Want algorithm to discover structure

2. **Arbitrary shapes**
   - Non-spherical clusters
   - Complex geometric patterns
   - Geographic/spatial data

3. **Noise is present**
   - Want to identify outliers
   - Noise points shouldn't form clusters

4. **Varying cluster sizes**
   - Some clusters large, others small
   - K-Means biases toward equal-sized clusters

### ❌ Don't Use DBSCAN When:

1. **Varying densities**
   - Clusters have different densities
   - HDBSCAN (hierarchical DBSCAN) better

2. **Very large datasets**
   - n > 100,000 points
   - Consider sampling or approximate methods

3. **High dimensions**
   - Curse of dimensionality affects distance metrics
   - Consider dimensionality reduction first

4. **Need cluster centroids**
   - DBSCAN doesn't compute centers
   - K-Means provides centroids directly

### Real-World Decision Framework

**Use K-Means if:**
- You know k (e.g., customer segments: small/medium/large)
- Spherical clusters are reasonable
- Need fast, scalable solution
- Want cluster centers for interpretation

**Use DBSCAN if:**
- Geographic/spatial data (cities, crime hot spots)
- Anomaly detection is important
- Complex, irregular shapes expected
- Don't know number of clusters

**Use Hierarchical if:**
- Want dendrogram visualization
- Need clusters at multiple scales
- Small dataset (< 10K points)

**Use GMM if:**
- Want soft (probabilistic) assignments
- Assume Gaussian clusters
- Need generative model

## Summary

🎯 **What We Learned:**

1. **DBSCAN** = Density-Based Spatial Clustering of Applications with Noise
2. **Core concepts**: ε-neighborhoods, core/border/noise points, density-reachability
3. **Algorithm**: Expand clusters from core points via density connection
4. **Parameters**: ε (neighborhood radius), MinPts (minimum density)
5. **K-distance plot**: Method to choose ε automatically
6. **Strengths**: Arbitrary shapes, noise detection, auto-discovers k
7. **Weaknesses**: Struggles with varying densities, parameter-sensitive

🔬 **Real-World Impact:**

- **Uber/Lyft**: Discover high-demand pickup zones automatically
- **Crime analysis**: Identify hot spots for police deployment
- **Astronomy**: Galaxy cluster discovery from star positions
- **Retail**: Customer location clustering for store placement

🚀 **Next Steps:**

In **Lesson 3B**, we'll:
- Use scikit-learn's production DBSCAN
- Apply to real geographic data
- Compare with HDBSCAN (hierarchical DBSCAN)
- Handle varying densities
- Build complete anomaly detection pipeline

---

**Key Takeaway**: DBSCAN shines when you have spatial data with **arbitrary shapes** and **unknown number of clusters**. Use k-distance plot to tune ε, start with MinPts = 2×dimensions. Perfect for geographic clustering and outlier detection!